# Common Setup

In [ ]:
!pip install ultralytics

In [ ]:
import pandas as pd
import shutil
import glob
from pathlib import Path
import os

from ultralytics import YOLO
import torch
from torchvision.utils import draw_bounding_boxes
import torchvision.transforms.functional as TF

from tqdm import tqdm

import matplotlib.pyplot as plt
import cv2
from PIL import Image

## Transform Dataset for Training

In [ ]:
# Root directory where the Kaggle dataset is mounted
KAGGLE_INPUT_ROOT = '/kaggle/input/datasets/zayeemb/birdsai'

# Where the YOLO-formatted data will be saved to
OUTPUT_DIR = '/kaggle/working/yolo_birdsai'

# BIRDSAI provides annotations in the standard MOT (Multiple Object Tracking) format.
# We map the columns so pandas knows how to parse the CSVs.
MOT_COLUMNS = [
    'frame_number', 'object_id', 'x', 'y', 'w', 'h', 
    'class_id', 'species', 'occlusion', 'noise'
]

# Map our YOLO splits to the raw dataset folders
# --- CHANGED: Both train and val point to TrainReal. Test points to TestReal. ---
SPLITS = {
    'train': os.path.join(KAGGLE_INPUT_ROOT, 'conservation_drones_train_real', 'TrainReal'),
    'val': os.path.join(KAGGLE_INPUT_ROOT, 'conservation_drones_train_real', 'TrainReal'),
    'test': os.path.join(KAGGLE_INPUT_ROOT, 'conservation_drones_test_real', 'TestReal')
}

def setup_directories(base_dir):
    """
    Creates the exact folder hierarchy required by YOLOv8.
    YOLO expects a strict split of images/ and labels/ directories.
    """
    # Added test directories to the folder creation list
    dirs = [
        f'{base_dir}/images/train', f'{base_dir}/images/val', f'{base_dir}/images/test',
        f'{base_dir}/labels/train', f'{base_dir}/labels/val', f'{base_dir}/labels/test'
    ]
    for d in dirs:
        os.makedirs(d, exist_ok=True)  # exist_ok=True prevents crashes if the folder already exists
    print(f"Created YOLO directory structure at {base_dir}")

def process_split(yolo_split_name, source_dir):
    """
    Converts MOT annotations to YOLO format, copies the corresponding images,
    and dynamically handles varying video resolutions.
    """
    annotations_dir = os.path.join(source_dir, 'annotations')
    images_dir = os.path.join(source_dir, 'images')
    
    csv_files = glob.glob(os.path.join(annotations_dir, '*.csv'))
    
    # 80/20 train/val split logic based on CSV (video) files
    csv_files.sort() # Sort to ensure the exact same deterministic 80/20 split happens every time
    split_index = int(len(csv_files) * 0.8)
    
    if yolo_split_name == 'train':
        csv_files = csv_files[:split_index]      # First 80% of TrainReal
    elif yolo_split_name == 'val':
        csv_files = csv_files[split_index:]      # Remaining 20% of TrainReal
    # If yolo_split_name == 'test', it bypasses the split and uses 100% of the TestReal folder
    
    print(f"Found {len(csv_files)} annotation files for {yolo_split_name} split in {source_dir}")
    
    # To avoid slow OS file searches inside the loop, we walk the directory once
    # and map filenames (e.g. '000000.jpg') to their full absolute paths.
    print("Indexing images...")
    image_lookup = {}
    for root, _, files in os.walk(images_dir):
        for file in files:
            if file.lower().endswith(('.jpg', '.png', '.jpeg')):
                parent_folder = os.path.basename(root)
                # Store both the nested structure (folder/file.jpg) and flat structure (file.jpg)
                image_lookup[f"{parent_folder}/{file}"] = os.path.join(root, file)
                image_lookup[file] = os.path.join(root, file)
    
    for csv_path in csv_files:
        video_name = Path(csv_path).stem  # Extracts the filename without the .csv extension
        
        # Load the annotations and filter out everything except Animals (0) and Humans (1)
        df = pd.read_csv(csv_path, names=MOT_COLUMNS)
        df = df[df['class_id'].isin([0, 1])] 
        
        # If this video has no valid targets, skip processing it entirely
        if df.empty:
            continue
            
        # Group by frame so we can write all bounding boxes for a single image into one YOLO text file
        grouped = df.groupby('frame_number')
        
        for frame_idx, group in tqdm(grouped, desc=f"Processing {video_name}", leave=False):
            
            # BIRDSAI uses a strict 10-digit zero-padded format for image names in some folders
            target_filename = f"{video_name}_{int(frame_idx):010d}.jpg"
            src_img_path = None
            
            nested_key = f"{video_name}/{target_filename}"
            flat_key = target_filename
            
            # Look up the absolute path from our pre-computed dictionary
            if nested_key in image_lookup:
                src_img_path = image_lookup[nested_key]
            elif flat_key in image_lookup:
                src_img_path = image_lookup[flat_key]
                    
            # If the CSV has an annotation but the image file doesn't exist, safely skip it
            if not src_img_path:
                continue 
            
            # Drone flights are recorded at different resolutions (e.g., 640x512 vs 640x480).
            # We MUST extract the true dimensions of this specific image to avoid bounding box drift.
            img = cv2.imread(src_img_path)
            if img is None:
                continue
            
            img_height, img_width = img.shape[:2]
            
            # Define output filenames (standardizing to a 6-digit padded format for YOLO)
            out_prefix = f"{video_name}_{int(frame_idx):06d}"
            dst_img_path = os.path.join(OUTPUT_DIR, 'images', yolo_split_name, f"{out_prefix}.jpg") 
            dst_label_path = os.path.join(OUTPUT_DIR, 'labels', yolo_split_name, f"{out_prefix}.txt")
            
            # Copy the physical image to the YOLO training folder
            shutil.copy(src_img_path, dst_img_path)
        
            # YOLO format: [class_id] [x_center] [y_center] [width] [height]
            # All geometric values must be normalized between 0.0 and 1.0
            with open(dst_label_path, 'w') as f:
                for _, row in group.iterrows():
                    class_id = int(row['class_id'])
                    
                    # Convert MOT top-left coordinates to YOLO center coordinates, then normalize
                    # We use max/min to strictly clamp the coordinates so boxes don't spill off the edge of the screen
                    x_center = max(0.0, min(1.0, (row['x'] + (row['w'] / 2)) / img_width))
                    y_center = max(0.0, min(1.0, (row['y'] + (row['h'] / 2)) / img_height))
                    w_norm = max(0.0, min(1.0, row['w'] / img_width))
                    h_norm = max(0.0, min(1.0, row['h'] / img_height))
                    
                    f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")

In [ ]:
setup_directories(OUTPUT_DIR)

for yolo_split, source_path in SPLITS.items():
    if os.path.exists(source_path):
        process_split(yolo_split, source_path)
    else:
        print(f"WARNING: Source path not found: {source_path}")
        
print("Dataset conversion complete! Ready for YOLO.")

In [ ]:
yaml_content = """
path: /kaggle/working/yolo_birdsai # The root directory of your processed dataset
train: images/train                # Train images (relative to 'path')
val: images/val                    # Validation images (relative to 'path')
test: images/test                  # Test images (relative to 'path')

# Classes dictionary
names:
  0: Animal
  1: Human
"""

# Write the YAML file to your working directory
yaml_path = '/kaggle/working/yolo_birdsai/dataset.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"Success! dataset.yaml created at: {yaml_path}")

## Directories Setup

In [ ]:
train_img_dir = "/kaggle/working/yolo_birdsai/images/train"
train_label_dir = "/kaggle/working/yolo_birdsai/labels/train"
test_img_dir = '/kaggle/working/yolo_birdsai/images/test'
test_label_dir = '/kaggle/working/yolo_birdsai/labels/test'
val_img_dir = "/kaggle/working/yolo_birdsai/images/val"
val_label_dir = "/kaggle/working/yolo_birdsai/labels/val"

# YOLO

In [ ]:

def train_baseline():
    yaml_path = '/kaggle/working/yolo_birdsai/dataset.yaml'
    output_dir = '/kaggle/working/runs/train'
    
    # Check if YAML exists before starting
    if not os.path.exists(yaml_path):
        raise FileNotFoundError(f"dataset.yaml not found at {yaml_path}. Did you run the generation script?")

    # We use YOLOv8 Nano (n) as discussed. It will auto-download the pre-trained weights.
    print("Loading YOLOv8n baseline model...")
    model = YOLO('yolov8m.pt') 
    
    print("Beginning training on Kaggle T4 GPU...")
    results = model.train(
        data=yaml_path,
        epochs=30,                  # 50 is a solid baseline. Reduce to 30 if Kaggle time is running short.
        imgsz=640,                  # Standard resolution
        batch=16,                   # 16 is perfectly safe for a Kaggle 16GB T4 GPU
        device=[0,1],                   # '0' explicitly forces the script to use the first GPU
        project=output_dir,         # Where to save all outputs
        name='baseline_birdsai',    # Name of the folder containing this specific run's data
        plots=True,                 # CRITICAL: Forces generation of the training curves for your report
        save=True                   # Saves the model weights (best.pt)
    )
    
    print(f"\n✅ Training complete! All deliverables saved to: {os.path.join(output_dir, 'baseline_birdsai')}")


## Train YOLO

In [ ]:
train_baseline()

## Quantitave Analysis YOLO

In [ ]:
yolo_weights = '/kaggle/input/models/zayeemb/yolo/pytorch/default/1/best.pt'
output_dir = '/kaggle/working/report_yolo_analysis'

In [ ]:
def test_evaluation_yolo(weights_path, conf_threshold=0.25):
    """
    Evaluates the trained YOLO model on the Test Set using torchmetrics
    to generate the exact Small/Medium/Large metrics needed for Table 2.
    """
    if not os.path.exists(weights_path):
        raise FileNotFoundError(f"YOLO Weights not found at {weights_path}!")

    print(f"\n--- Loading YOLOv8 Model ---")
    model = YOLO(weights_path)
    
    # Setup Dataset for Ground Truth
    # We use our custom dataset class just to easily load the scaled Ground Truth boxes
    test_dataset = BIRDSAIYOLODataset(test_img_dir, test_label_dir) 
    
    # 3. Setup Metrics (IoU=0.5)
    metric_animals = MeanAveragePrecision(iou_type="bbox", iou_thresholds=[0.5])
    metric_humans = MeanAveragePrecision(iou_type="bbox", iou_thresholds=[0.5])
    
    print("\n--- Running Quantitative Evaluation (Calculating Table 2 Metrics) ---")
    
    for idx in tqdm(range(len(test_dataset)), desc="Evaluating YOLO Test Set"):
        _, target = test_dataset[idx] 
        img_name = test_dataset.img_names[idx]
        img_path = os.path.join(test_img_dir, img_name)
        
        # 1. Forward Pass (YOLO)
        results = model(img_path, verbose=False)[0]
        
        # 2. Extract YOLO Predictions
        # YOLO returns absolute coordinates, which matches torchmetrics requirements
        p_boxes = results.boxes.xyxy.cpu() 
        p_scores = results.boxes.conf.cpu()
        
        # CRITICAL ALIGNMENT: YOLO predicts 0 (Animal) and 1 (Human). 
        # Our ground truth from the Dataset uses 1 and 2. We add 1 to match them.
        p_labels = results.boxes.cls.cpu().int() + 1
        
        # 3. Extract Ground Truth
        t_boxes = target['boxes'].cpu()
        t_labels = target['labels'].cpu()
        
        # --- Filter for Animals (Class 1) ---
        a_pred_mask = p_labels == 1
        a_targ_mask = t_labels == 1
        
        metric_animals.update(
            [{"boxes": p_boxes[a_pred_mask], "scores": p_scores[a_pred_mask], "labels": p_labels[a_pred_mask]}],
            [{"boxes": t_boxes[a_targ_mask], "labels": t_labels[a_targ_mask]}]
        )
        
        # --- Filter for Humans (Class 2) ---
        h_pred_mask = p_labels == 2
        h_targ_mask = t_labels == 2
        
        metric_humans.update(
            [{"boxes": p_boxes[h_pred_mask], "scores": p_scores[h_pred_mask], "labels": p_labels[h_pred_mask]}],
            [{"boxes": t_boxes[h_targ_mask], "labels": t_labels[h_targ_mask]}]
        )

    # Compute Final Scores
    res_animals = metric_animals.compute()
    res_humans = metric_humans.compute()
    
    print("\n" + "="*50)
    print("📋 YOLO TABLE 2 METRICS REPORT (mAP @ 0.5)")
    print("="*50)
    print(f"SA (Small Animals):  {res_animals['map_small'].item():.4f}")
    print(f"MA (Medium Animals): {res_animals['map_medium'].item():.4f}")
    print(f"LA (Large Animals):  {res_animals['map_large'].item():.4f}")
    print(f"Animals (Overall):   {res_animals['map_50'].item():.4f}")
    print("-" * 50)
    print(f"SH (Small Humans):   {res_humans['map_small'].item():.4f}")
    print(f"MH (Medium Humans):  {res_humans['map_medium'].item():.4f}")
    print(f"LH (Large Humans):   {res_humans['map_large'].item():.4f}")
    print(f"Humans (Overall):    {res_humans['map_50'].item():.4f}")
    print("="*50)

    # 4. Qualitative Check (Visual Predictions)
    print("\n--- Generating Visual Predictions ---")
    output_dir = '/kaggle/working/predict_yolo_baseline'
    os.makedirs(output_dir, exist_ok=True)
    
    # Grab the first 10 images for visualization using YOLO's built-in plotter
    sample_imgs = [os.path.join(test_img_dir, test_dataset.img_names[i]) for i in range(10)]
    
    model.predict(
        source=sample_imgs,
        save=True,                 
        project='/kaggle/working',
        name='predict_yolo_baseline',
        exist_ok=True, # Prevent it from making new folders like predict_yolo_baseline2
        conf=conf_threshold                  
    )
    
    print(f"\n✅ YOLO Evaluation complete!")
    print(f"Visual predictions saved to: {output_dir}")

if __name__ == '__main__':
    test_evaluation_yolo(yolo_weights)

## Qualitative Analysis YOLO

In [ ]:
def generate_yolo_success_failure_visuals(num_samples=15, conf_threshold=0.25):
    """
    Overlays Ground Truth (Green) and YOLO Predictions (Red) on the same image
    to easily identify YOLO's successes and failures for the report.
    """
    # Setup Paths
    
    os.makedirs(output_dir, exist_ok=True)
    
    if not os.path.exists(yolo_weights):
        raise FileNotFoundError(f"Cannot find YOLO weights at {yolo_weights}")
    
    print("Loading YOLOv8 Baseline Model...")
    model = YOLO(yolo_weights)
    
    # Load dataset just to easily fetch the perfectly scaled Ground Truth boxes
    test_dataset = BIRDSAIYOLODataset(test_img_dir, test_label_dir)
    
    # Class Maps: YOLO outputs 0/1 natively, but our Dataset ground truth shifted them to 1/2
    gt_class_names = {1: "Animal", 2: "Human"}  
    yolo_class_names = {0: "Animal", 1: "Human"} 
    
    print(f"Generating {num_samples} YOLO evaluation images...")
    
    # Pick random images from the test set
    indices = random.sample(range(len(test_dataset)), min(num_samples, len(test_dataset)))
    
    for idx in indices:
        img_tensor, target = test_dataset[idx]
        img_name = test_dataset.img_names[idx]
        img_path = os.path.join(test_img_dir, img_name)
        
        # 1. Get Ground Truth (GREEN)
        gt_boxes = target['boxes']
        gt_labels = target['labels']
        gt_strings = [f"GT {gt_class_names[l.item()]}" for l in gt_labels]
        
        # 2. Get YOLO Predictions (RED)
        results = model(img_path, verbose=False)[0]
        
        # Filter by confidence threshold
        keep = results.boxes.conf.cpu() > conf_threshold
        pred_boxes = results.boxes.xyxy.cpu()[keep]
        pred_labels = results.boxes.cls.cpu().int()[keep]
        pred_scores = results.boxes.conf.cpu()[keep]
        
        # Format strings for the predictions
        pred_strings = [f"Pred {yolo_class_names[l.item()]}: {s:.2f}" for l, s in zip(pred_labels, pred_scores)]
        
        # 3. Draw the Image
        # Convert tensor to uint8 image for drawing
        draw_tensor = (img_tensor * 255).to(torch.uint8)
        
        # Draw Ground Truth in GREEN
        if len(gt_boxes) > 0:
            draw_tensor = draw_bounding_boxes(draw_tensor, gt_boxes, labels=gt_strings, colors="green", width=3, font_size=16)
            
        # Draw Predictions in RED
        if len(pred_boxes) > 0:
            draw_tensor = draw_bounding_boxes(draw_tensor, pred_boxes, labels=pred_strings, colors="red", width=3, font_size=16)
            
        # Save to disk
        img_cv = draw_tensor.permute(1, 2, 0).numpy()
        img_cv = cv2.cvtColor(img_cv, cv2.COLOR_RGB2BGR)
        save_path = os.path.join(output_dir, f"yolo_analysis_{img_name}")
        cv2.imwrite(save_path, img_cv)

    print(f"\n✅ YOLO Visuals saved to: {output_dir}")
    print("🟢 GREEN = Ground Truth  |  🔴 RED = YOLO Prediction")

if __name__ == '__main__':
    # You can change conf_threshold to 0.10 if you want to see if YOLO is at least 
    # slightly confident about the missed targets
    generate_yolo_success_failure_visuals(num_samples=15, conf_threshold=0.25)

# Faster RCNN


In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.models._utils import IntermediateLayerGetter
from torchvision.ops import FeaturePyramidNetwork, MultiScaleRoIAlign
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import torch.nn.functional as F


import os
from pathlib import Path
import glob
import shutil

from PIL import Image
import cv2

from tqdm.auto import tqdm
import math
import pandas as pd
import matplotlib.pyplot as plt

## Dataloader for Faster RCNN

In [ ]:
class BIRDSAIYOLODataset(Dataset):
    """
    A custom PyTorch Dataset that reads YOLO-formatted .txt labels
    and converts them on-the-fly to the Faster R-CNN dictionary format.
    """
    def __init__(self, img_dir, label_dir):
        self.img_dir = img_dir
        self.label_dir = label_dir
        
        # Get a list of all image filenames
        self.img_names = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        self.img_names.sort() # Ensure consistent ordering

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_name = self.img_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        # Find the corresponding YOLO .txt file
        label_name = os.path.splitext(img_name)[0] + '.txt'
        label_path = os.path.join(self.label_dir, label_name)

        # 1. Load Image and get true dimensions
        img = Image.open(img_path).convert("RGB")
        img_w, img_h = img.size
        
        # Convert PIL Image to PyTorch Tensor (Channels x Height x Width, scaled 0 to 1)
        img_tensor = TF.to_tensor(img)

        boxes = []
        labels = []

        # Parse the YOLO .txt file (if it exists and isn't empty)
        if os.path.exists(label_path) and os.path.getsize(label_path) > 0:
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5: continue
                    
                    yolo_class = int(parts[0])
                    x_center, y_center, w_norm, h_norm = map(float, parts[1:])
                    
                    # Convert YOLO (center, w, h) to PyTorch (xmin, ymin, xmax, ymax)
                    x_min = (x_center - (w_norm / 2)) * img_w
                    y_min = (y_center - (h_norm / 2)) * img_h
                    x_max = (x_center + (w_norm / 2)) * img_w
                    y_max = (y_center + (h_norm / 2)) * img_h
                    
                    # Clamp boxes strictly to image borders just in case
                    x_min = max(0.0, min(x_min, float(img_w)))
                    y_min = max(0.0, min(y_min, float(img_h)))
                    x_max = max(0.0, min(x_max, float(img_w)))
                    y_max = max(0.0, min(y_max, float(img_h)))
                    
                    # Only keep boxes that have an actual area > 0
                    if x_max > x_min and y_max > y_min:
                        boxes.append([x_min, y_min, x_max, y_max])
                        
                        # Shift labels! (0->1, 1->2)
                        labels.append(yolo_class + 1)

        # Format for PyTorch
        if len(boxes) > 0:
            boxes_tensor = torch.tensor(boxes, dtype=torch.float32)
            labels_tensor = torch.tensor(labels, dtype=torch.int64)
        else:
            # Faster RCNN expects a specific tensor shape for empty images
            boxes_tensor = torch.empty((0, 4), dtype=torch.float32)
            labels_tensor = torch.empty((0,), dtype=torch.int64)

        target = {
            "boxes": boxes_tensor,
            "labels": labels_tensor,
            "image_id": torch.tensor([idx])
        }

        return img_tensor, target

## Faster RCNN Backbone Class

In [ ]:
def collate_fn(batch):
    """
    Tells PyTorch how to batch dictionaries of varying sizes together.
    """
    return tuple(zip(*batch))

In [ ]:

class CustomResNet50FPN(nn.Module):
    """
    A custom backbone that extracts multi-scale features from a standard 
    ResNet50 and passes them through an explicit Feature Pyramid Network.
    """
    def __init__(self):
        super().__init__()
        
        # Load the raw ResNet-50
        print("Loading raw ResNet-50...")
        resnet = resnet50(weights=ResNet50_Weights.DEFAULT)
        
        # Extract intermediate layers
        # ResNet produces features with these channel depths at these layers
        # layer1: 256, layer2: 512, layer3: 1024, layer4: 2048
        return_layers = {
            'layer1': '0', # Maps 'layer1' output to name '0'
            'layer2': '1', # Maps 'layer2' output to name '1'
            'layer3': '2', # Maps 'layer3' output to name '2'
            'layer4': '3'  # Maps 'layer4' output to name '3'
        }
        self.body = IntermediateLayerGetter(resnet, return_layers=return_layers)
        
        # Explicitly construct the Feature Pyramid Network
        # It takes the channels from the ResNet layers and unifies them to 256 channels
        in_channels_list = [256, 512, 1024, 2048]
        out_channels = 256
        self.fpn = FeaturePyramidNetwork(
            in_channels_list=in_channels_list,
            out_channels=out_channels
        )
        
        # REQUIRED by FasterRCNN: tell it how many channels we are outputting
        self.out_channels = out_channels

    def forward(self, x):
        # Pass image through ResNet to get the 4 feature maps
        x = self.body(x)
        # Pass the 4 feature maps through the FPN
        x = self.fpn(x)
        return x

def build_custom_frcnn_base(num_classes=3):
    """
    Builds the full Faster R-CNN using our custom explicit backbone.
    """
    # Initialize our custom backbone
    backbone = CustomResNet50FPN()
    
    # Define anchors for the 4 FPN levels
    # Since we have 4 levels from our IntermediateLayerGetter, we need 4 tuples
    anchor_sizes = ((32,), (64,), (128,), (256,))
    aspect_ratios = ((0.5, 1.0, 2.0),) * len(anchor_sizes)
    
    print("Initializing explicit RPN Anchor Generator...")
    rpn_anchor_generator = AnchorGenerator(
        sizes=anchor_sizes, 
        aspect_ratios=aspect_ratios
    )
    
    # Define the RoI Pooler to pull from all 4 FPN levels
    print("Initializing explicit MultiScale RoI Align...")
    roi_pooler = MultiScaleRoIAlign(
        featmap_names=['0', '1', '2', '3'], 
        output_size=7,
        sampling_ratio=2
    )
    
    # Assemble the final model
    print("Assembling Faster R-CNN architecture...")
    model = FasterRCNN(
        backbone=backbone,
        num_classes=num_classes,
        rpn_anchor_generator=rpn_anchor_generator,
        box_roi_pool=roi_pooler
    )
    
    return model

# Quick execution check
if __name__ == "__main__":
    model_ce = build_custom_frcnn_base(num_classes=3)
    print("\n✅ Custom FR-CE Model successfully assembled from raw primitives!")

## Different Loses for Faster RCNN

In [ ]:
import torch.nn.functional as F
from torchvision.models.detection import roi_heads

# Define the Focal Loss Math
def focal_loss(class_logits, labels, alpha=0.25, gamma=2.0):
    ce_loss = F.cross_entropy(class_logits, labels, reduction='none')
    pt = torch.exp(-ce_loss)
    focal_loss = alpha * (1 - pt)**gamma * ce_loss
    return focal_loss.mean()

# The Monkey-Patch Setup
original_fastrcnn_loss = roi_heads.fastrcnn_loss

def custom_focal_fastrcnn_loss(class_logits, box_regression, labels, regression_targets):
    labels_cat = torch.cat(labels, dim=0)
    classification_loss = focal_loss(class_logits, labels_cat, alpha=0.25, gamma=2.0)
    _, original_box_loss = original_fastrcnn_loss(class_logits, box_regression, labels, regression_targets)
    return classification_loss, original_box_loss

# Build the FR-FL Model
def build_frcnn_focal_loss(num_classes=3):
    model = build_custom_frcnn_base(num_classes=num_classes)
    roi_heads.fastrcnn_loss = custom_focal_fastrcnn_loss
    print("Focal Loss successfully injected into RoI Heads!")
    return model

## Training Scripts for Faster RCNN

In [ ]:
def train_one_epoch(model, optimizer, data_loader, device, epoch):
    """
    Trains the model for a single epoch over all batches with a live progress bar.
    """
    model.train() # Put model in training mode
    total_loss_this_epoch = 0.0
    
    # Wrap the data loader in tqdm! 
    # desc= sets the label, leave=True keeps the finished bar on screen
    loop = tqdm(data_loader, desc=f"Epoch [{epoch}]", leave=True)
    
    # Loop over every batch of images
    for batch_idx, (images, targets) in enumerate(loop):
        
        # Move data to the GPU
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        # Forward Pass
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        # Failsafe: If loss explodes to infinity, stop training
        if not math.isfinite(losses.item()):
            print(f"Loss is {losses.item()}, stopping training")
            break
            
        # Backpropagation
        optimizer.zero_grad() # Clear old gradients
        losses.backward()     # Calculate new gradients
        optimizer.step()      # Update the weights
        
        total_loss_this_epoch += losses.item()
        
        loop.set_postfix(loss=f"{losses.item():.4f}")
            
    # Return the average loss for this epoch so we can plot it later
    return total_loss_this_epoch / len(data_loader)

In [ ]:
def run_training_experiment(experiment_type="CE", num_epochs=10):
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    print(f"Training on device: {device}")
    
    dataset = BIRDSAIYOLODataset(train_img_dir, train_label_dir)
    
    # collate_fn is our custom tuple-packer from earlier!
    data_loader = DataLoader(
        dataset, batch_size=4, shuffle=True, 
        num_workers=2, collate_fn=collate_fn 
    )
    
    # Initialize the specific Model (CE vs Focal Loss)
    if experiment_type == "CE":
        print("\n--- Starting FR-CE (Cross Entropy) Experiment ---")
        model = build_custom_frcnn_base(num_classes=3)
    else:
        print("\n--- Starting FR-FL (Focal Loss) Experiment ---")
        model = build_frcnn_focal_loss(num_classes=3)
        
    model.to(device)
    
    # Setup the Optimizer (Standard SGD with momentum is best for Faster R-CNN)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
    
    # The Epoch Loop
    loss_history = []
    
    for epoch in range(1, num_epochs + 1):
        epoch_loss = train_one_epoch(model, optimizer, data_loader, device, epoch)
        loss_history.append(epoch_loss)
        print(f"*** End of Epoch {epoch} - Average Loss: {epoch_loss:.4f} ***\n")
        
    # Save the final weights
    save_path = f'/kaggle/working/frcnn_{experiment_type.lower()}_weights.pth'
    torch.save(model.state_dict(), save_path)
    print(f"✅ Training Complete! Weights saved to {save_path}")
    
    return loss_history


## Train with CE Loss

In [ ]:
# Start with a short 5-epoch run just to make sure the pipeline doesn't crash
ce_losses = run_training_experiment(experiment_type="CE", num_epochs=5)

# Plot the quick curve
plt.plot(range(1, 6), ce_losses, marker='o', label='FR-CE Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Curve: Faster R-CNN (Cross Entropy)')
plt.legend()
plt.show()

## Train with FocalLoss

In [ ]:
# Start with a short 5-epoch run just to make sure the pipeline doesn't crash
ce_losses = run_training_experiment(experiment_type="FL", num_epochs=5)

# Plot the quick curve
plt.plot(range(1, 6), ce_losses, marker='o', label='FR-CE Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Curve: Faster R-CNN (Cross Entropy)')
plt.legend()
plt.show()

## Quantitative Analysis of Faster RCNN

In [ ]:
weights_path = '/kaggle/input/models/zayeemb/faster-rcnn/pytorch/default/1/frcnn_ce_weights.pth'

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchvision.utils import draw_bounding_boxes

def test_evaluation_frcnn(experiment_type="CE", conf_threshold=0.5):
    """
    Evaluates the trained Faster R-CNN model on the Test Set.
    Generates quantitative COCO metrics and qualitative bounding box images.
    """
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    print(f"Evaluating on device: {device}")
    
    # 1. Load the Model and Weights
    if experiment_type == "CE":
        print("\n--- Loading FR-CE (Cross Entropy) Model ---")
        model = build_custom_frcnn_base(num_classes=3)
    else:
        print("\n--- Loading FR-FL (Focal Loss) Model ---")
        model = build_frcnn_focal_loss(num_classes=3)
        
    if not os.path.exists(weights_path):
        raise FileNotFoundError(f"Weights not found at {weights_path}!")
        
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.to(device)
    model.eval() # CRITICAL: Put model in evaluation mode
    
    # Load the Test Dataset
    test_dataset = BIRDSAIYOLODataset(test_img_dir, test_label_dir)
    test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)
    
    # Setup Metrics (We use IoU=0.5 exactly as your rubric requested)
    # We create two separate metrics to isolate Animal vs Human scales perfectly
    metric_animals = MeanAveragePrecision(iou_type="bbox", iou_thresholds=[0.5])
    metric_humans = MeanAveragePrecision(iou_type="bbox", iou_thresholds=[0.5])
    
    print("\n--- Running Quantitative Evaluation (Calculating Table 2 Metrics) ---")
    
    with torch.no_grad(): # No gradients needed for testing
        for images, targets in tqdm(test_loader, desc="Evaluating Test Set"):
            images = list(img.to(device) for img in images)
            
            # Forward Pass
            outputs = model(images)
            
            # Format predictions and targets for torchmetrics
            for i in range(len(outputs)):
                pred = outputs[i]
                target = targets[i]
                
                # Move to CPU to prevent RAM leaks
                p_boxes = pred['boxes'].cpu()
                p_scores = pred['scores'].cpu()
                p_labels = pred['labels'].cpu()
                
                t_boxes = target['boxes'].cpu()
                t_labels = target['labels'].cpu()
                
                # --- Filter for Animals (Class 1) ---
                a_pred_mask = p_labels == 1
                a_targ_mask = t_labels == 1
                
                metric_animals.update(
                    [{"boxes": p_boxes[a_pred_mask], "scores": p_scores[a_pred_mask], "labels": p_labels[a_pred_mask]}],
                    [{"boxes": t_boxes[a_targ_mask], "labels": t_labels[a_targ_mask]}]
                )
                
                # --- Filter for Humans (Class 2) ---
                h_pred_mask = p_labels == 2
                h_targ_mask = t_labels == 2
                
                metric_humans.update(
                    [{"boxes": p_boxes[h_pred_mask], "scores": p_scores[h_pred_mask], "labels": p_labels[h_pred_mask]}],
                    [{"boxes": t_boxes[h_targ_mask], "labels": t_labels[h_targ_mask]}]
                )

    # Compute Final Scores
    res_animals = metric_animals.compute()
    res_humans = metric_humans.compute()
    
    print("\n" + "="*50)
    print("📋 TABLE 2 METRICS REPORT (mAP @ 0.5)")
    print("="*50)
    print(f"SA (Small Animals):  {res_animals['map_small'].item():.4f}")
    print(f"MA (Medium Animals): {res_animals['map_medium'].item():.4f}")
    print(f"LA (Large Animals):  {res_animals['map_large'].item():.4f}")
    print(f"Animals (Overall):   {res_animals['map_50'].item():.4f}")
    print("-" * 50)
    print(f"SH (Small Humans):   {res_humans['map_small'].item():.4f}")
    print(f"MH (Medium Humans):  {res_humans['map_medium'].item():.4f}")
    print(f"LH (Large Humans):   {res_humans['map_large'].item():.4f}")
    print(f"Humans (Overall):    {res_humans['map_50'].item():.4f}")
    print("="*50)

    # 4. Qualitative Check (Visual Predictions)
    print("\n--- Generating Visual Predictions ---")
    output_dir = f'/kaggle/working/predict_frcnn_{experiment_type.lower()}'
    os.makedirs(output_dir, exist_ok=True)
    
    # Grab the first 10 images for visualization
    sample_imgs = []
    sample_names = test_dataset.img_names[:10]
    for i in range(10):
        img_tensor, _ = test_dataset[i]
        sample_imgs.append(img_tensor.to(device))
        
    with torch.no_grad():
        preds = model(sample_imgs)
        
    for i, pred in enumerate(preds):
        # Filter by confidence threshold
        keep = pred['scores'] > conf_threshold
        boxes = pred['boxes'][keep].cpu()
        labels = pred['labels'][keep].cpu()
        scores = pred['scores'][keep].cpu()
        
        # Format labels: 'Animal: 0.95'
        class_names = {1: "Animal", 2: "Human"}
        str_labels = [f"{class_names[l.item()]}: {s:.2f}" for l, s in zip(labels, scores)]
        
        # Convert image tensor back to standard byte format for drawing
        img_draw = (sample_imgs[i].cpu() * 255).to(torch.uint8)
        
        # Draw the boxes
        if len(boxes) > 0:
            img_draw = draw_bounding_boxes(img_draw, boxes, labels=str_labels, colors="red", width=3, font_size=20)
            
        # Save it
        img_cv = img_draw.permute(1, 2, 0).numpy() # Convert from (C,H,W) to (H,W,C)
        img_cv = cv2.cvtColor(img_cv, cv2.COLOR_RGB2BGR)
        cv2.imwrite(os.path.join(output_dir, sample_names[i]), img_cv)

    print(f"✅ Sanity check complete!")
    print(f"Visual predictions saved to: {output_dir}")


if __name__ == '__main__':
    # Run this for Cross Entropy first
    test_evaluation_frcnn(experiment_type="CE", conf_threshold=0.5)
    test_evaluation_frcnn(experiment_type="FL", conf_threshold=0.5)
    
    # Once you train your Focal Loss model, change to:
    # test_evaluation_frcnn(experiment_type="FL", conf_threshold=0.5)

## Qualitative Analysis of RCNN

In [1]:
import torch
import cv2
import matplotlib.pyplot as plt
import os
import random
from torchvision.utils import draw_bounding_boxes
import torchvision.transforms.functional as TF

def generate_success_failure_visuals(num_samples=10, experiment_type="FL", conf_threshold=0.5):
    """
    Overlays Ground Truth (Green) and Predictions (Red) on the same image
    to easily identify successes and failures for the report.
    """
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    output_dir = f'/kaggle/working/report_individual_analysis'
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Loading FR-{experiment_type} Model for Individual Analysis...")
    if experiment_type == "CE":
        model = build_custom_frcnn_base(num_classes=3)
    else:
        model = build_frcnn_focal_loss(num_classes=3)
        
    weights_path = f'/kaggle/working/frcnn_{experiment_type.lower()}_weights.pth'
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.to(device)
    model.eval()
    
    # Load Test Data
    test_dataset = BIRDSAIYOLODataset(test_img_dir, test_label_dir)
    
    class_names = {1: "Animal", 2: "Human"}
    
    print(f"Generating {num_samples} evaluation images...")
    
    # Pick random images from the test set
    indices = random.sample(range(len(test_dataset)), min(num_samples, len(test_dataset)))
    
    for idx in indices:
        img_tensor, target = test_dataset[idx]
        img_name = test_dataset.img_names[idx]
        
        # 1. Get Ground Truth
        gt_boxes = target['boxes']
        gt_labels = target['labels']
        gt_strings = [f"GT {class_names[l.item()]}" for l in gt_labels]
        
        # 2. Get Predictions
        with torch.no_grad():
            pred = model([img_tensor.to(device)])[0]
            
        keep = pred['scores'] > conf_threshold
        pred_boxes = pred['boxes'][keep].cpu()
        pred_labels = pred['labels'][keep].cpu()
        pred_scores = pred['scores'][keep].cpu()
        pred_strings = [f"Pred {class_names[l.item()]}: {s:.2f}" for l, s in zip(pred_labels, pred_scores)]
        
        # 3. Draw the Image
        # Convert tensor to uint8 image for drawing
        draw_tensor = (img_tensor * 255).to(torch.uint8)
        
        # Draw Ground Truth in GREEN
        if len(gt_boxes) > 0:
            draw_tensor = draw_bounding_boxes(draw_tensor, gt_boxes, labels=gt_strings, colors="green", width=3, font_size=16)
            
        # Draw Predictions in RED
        if len(pred_boxes) > 0:
            draw_tensor = draw_bounding_boxes(draw_tensor, pred_boxes, labels=pred_strings, colors="red", width=3, font_size=16)
            
        # Save to disk
        img_cv = draw_tensor.permute(1, 2, 0).numpy()
        img_cv = cv2.cvtColor(img_cv, cv2.COLOR_RGB2BGR)
        save_path = os.path.join(output_dir, f"analysis_{img_name}")
        cv2.imwrite(save_path, img_cv)

    print(f"\n✅ Visuals saved to: {output_dir}")
    print("🟢 GREEN = Ground Truth  |  🔴 RED = Model Prediction")

Loading FR-FL Model...


NameError: name 'build_frcnn_focal_loss' is not defined

### Option B

In [ ]:
def generate_frcnn_grid_visuals(num_samples=15, cols=3, experiment_type="FL", conf_threshold=0.5):
    """
    Overlays Ground Truth (Green) and Predictions (Red) on the same image,
    outputting all samples into a single, clean matplotlib grid for the report.
    """
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    output_dir = f'/kaggle/working/report_frcnn_analysis'
    os.makedirs(output_dir, exist_ok=True)
    
    weights_path = f'/kaggle/input/models/zaiemgulzar/frcnn/pytorch/default/1/frcnn_{experiment_type.lower()}_weights.pth'
    
    if not os.path.exists(weights_path):
        print(f"Error: Weights not found at {weights_path}. Did you train the {experiment_type} model?")
        return
        
    print(f"Loading FR-{experiment_type} Model for Individual Analysis...")
    
    # Assumes these functions are defined in a previous cell!
    if experiment_type == "CE":
        model = build_custom_frcnn_base(num_classes=3)
    else:
        model = build_frcnn_focal_loss(num_classes=3)
        
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.to(device)
    model.eval()
    
    test_dataset = BIRDSAIYOLODataset(test_img_dir, test_label_dir)
    class_names = {1: "Animal", 2: "Human"}
    
    print(f"Generating a grid of {num_samples} evaluation images...")
    
    indices = random.sample(range(len(test_dataset)), min(num_samples, len(test_dataset)))
    
    # --- GRID SETUP ---
    rows = math.ceil(num_samples / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 8, rows * 6))
    axes = axes.flatten()
    
    for i, idx in enumerate(indices):
        img_tensor, target = test_dataset[idx]
        img_name = test_dataset.img_names[idx]
        
        # 1. Get Ground Truth (GREEN)
        gt_boxes = target['boxes']
        gt_labels = target['labels']
        gt_strings = [f"GT {class_names[l.item()]}" for l in gt_labels]
        
        # 2. Get Predictions (RED)
        with torch.no_grad():
            pred = model([img_tensor.to(device)])[0]
            
        keep = pred['scores'] > conf_threshold
        pred_boxes = pred['boxes'][keep].cpu()
        pred_labels = pred['labels'][keep].cpu()
        pred_scores = pred['scores'][keep].cpu()
        pred_strings = [f"Pred {class_names[l.item()]}: {s:.2f}" for l, s in zip(pred_labels, pred_scores)]
        
        # 3. Draw BOTH on the exact same image
        draw_tensor = (img_tensor * 255).to(torch.uint8)
        
        if len(gt_boxes) > 0:
            draw_tensor = draw_bounding_boxes(draw_tensor, gt_boxes, labels=gt_strings, colors="green", width=3, font_size=16)
        if len(pred_boxes) > 0:
            draw_tensor = draw_bounding_boxes(draw_tensor, pred_boxes, labels=pred_strings, colors="red", width=3, font_size=16)
            
        # 4. Place image into the specific grid slot
        img_rgb = draw_tensor.permute(1, 2, 0).numpy()
        axes[i].imshow(img_rgb)
        axes[i].set_title(img_name, fontsize=12, fontweight='bold')
        axes[i].axis('off')

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
        
    plt.tight_layout()
    
    # Add a master title
    model_name = "Focal Loss" if experiment_type == "FL" else "Cross Entropy"
    fig.suptitle(f"Faster R-CNN ({model_name}) Analysis\nGreen = Ground Truth | Red = Prediction (Conf > {conf_threshold})", 
                 fontsize=20, fontweight='bold', y=1.05)
    
    # Display in Notebook
    plt.show()
    
    # Save the massive grid as a single high-res image
    save_path = os.path.join(output_dir, f"frcnn_{experiment_type.lower()}_grid.png")
    fig.savefig(save_path, bbox_inches='tight', dpi=200)
    plt.close(fig)

    print(f"\n✅ FR-CNN Master grid successfully generated and saved to: {save_path}")

if __name__ == '__main__':
    # You can change experiment_type to "CE" to visualize the Cross Entropy baseline instead!
    generate_frcnn_grid_visuals(num_samples=15, cols=3, experiment_type="FL", conf_threshold=0.5)

## Side by Side Visual Comparison of YOLO and Faster RCNN

In [ ]:
def generate_comparison_figure(image_paths, yolo_path, frcnn_path, output_dir):
    """
    Runs both YOLO and Faster R-CNN on specific images, displays them inline 
    in the Kaggle cell, and saves the side-by-side comparison plots for your report.
    """
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    os.makedirs(output_dir, exist_ok=True)
    
    print("Loading YOLOv8 Baseline...")
    yolo_model = YOLO(yolo_path)
    
    print("Loading Faster R-CNN Model...")
    frcnn_model = build_frcnn_focal_loss(num_classes=3) # Assuming FR-FL is your best model
    frcnn_model.load_state_dict(torch.load(frcnn_path, map_location=device))
    frcnn_model.to(device)
    frcnn_model.eval()
    
    class_names = {1: "Animal", 2: "Human"}
    
    for img_path in image_paths:
        if not os.path.exists(img_path):
            print(f"Skipping {img_path}, not found.")
            continue
            
        img_name = os.path.basename(img_path)
        print(f"Processing {img_name}...")
        
        # --- 1. YOLO INFERENCE ---
        yolo_res = yolo_model(img_path, conf=0.25, verbose=False)[0]
        # YOLO's built-in plotter returns a BGR numpy array
        yolo_img_bgr = yolo_res.plot() 
        yolo_img_rgb = cv2.cvtColor(yolo_img_bgr, cv2.COLOR_BGR2RGB)
        
        # --- 2. FASTER R-CNN INFERENCE ---
        pil_img = Image.open(img_path).convert("RGB")
        img_tensor = TF.to_tensor(pil_img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            frcnn_res = frcnn_model(img_tensor)[0]
            
        # Filter FRCNN by confidence
        keep = frcnn_res['scores'] > 0.05
        boxes = frcnn_res['boxes'][keep].cpu()
        labels = frcnn_res['labels'][keep].cpu()
        scores = frcnn_res['scores'][keep].cpu()
        
        str_labels = [f"{class_names[l.item()]}: {s:.2f}" for l, s in zip(labels, scores)]
        
        # Draw FRCNN boxes
        draw_tensor = (img_tensor.squeeze(0).cpu() * 255).to(torch.uint8)
        if len(boxes) > 0:
            draw_tensor = draw_bounding_boxes(draw_tensor, boxes, labels=str_labels, colors="red", width=3, font_size=20)
        frcnn_img_rgb = draw_tensor.permute(1, 2, 0).numpy()
        
        # --- 3. PLOT SIDE-BY-SIDE ---
        fig, axes = plt.subplots(1, 2, figsize=(20, 10))
        
        axes[0].imshow(yolo_img_rgb)
        axes[0].set_title("YOLOv8 (Baseline)", fontsize=18, fontweight='bold')
        axes[0].axis('off')
        
        axes[1].imshow(frcnn_img_rgb)
        axes[1].set_title("Faster R-CNN + FPN (Focal Loss)", fontsize=18, fontweight='bold')
        axes[1].axis('off')
        
        plt.tight_layout()
        save_path = os.path.join(output_dir, f"comparison_{img_name}")
        
        # Save it to disk first
        plt.savefig(save_path, bbox_inches='tight', dpi=150)
        
        # --- THE FIX: Display the figure directly in the Kaggle output ---
        plt.show()
        
        # Close the specific figure to free up RAM
        plt.close(fig)
        
    print(f"\n✅ All comparison figures displayed above and saved to: {output_dir}")

In [ ]:
import random

FRCNN_WEIGHTS = "/kaggle/input/models/zayeemb/faster-rcnn/pytorch/default/1/frcnn_ce_weights.pth"
YOLO_WEIGHTS = "/kaggle/input/models/zayeemb/yolo/pytorch/default/1/best.pt"

test_dir = '/kaggle/working/yolo_birdsai/images/test'

easy_videos = ['0000000352_0000000000'] 
hard_videos = ['0000000058_0000000000'] 

def get_images_from_videos(video_list, num_samples=5):
    """Finds all frames belonging to specific videos and samples a few."""
    collected_frames = []
    for video_name in video_list:
        # We use a wildcard (*) to find all frames belonging to this specific video
        search_pattern = os.path.join(test_dir, f"{video_name}_*.jpg")
        frames = glob.glob(search_pattern)
        collected_frames.extend(frames)
        
    if len(collected_frames) == 0:
        print(f" Warning: Found 0 images for videos: {video_list}")
        return []
        
    # Randomly pick the requested number of frames from the matched videos
    return random.sample(collected_frames, min(num_samples, len(collected_frames)))

print("Gathering Easy Images...")
# Let's grab 3 random frames from your easy videos
easy_images = get_images_from_videos(easy_videos, num_samples=3)

print("Gathering Hard Images...")
# Let's grab 2 random frames from your hard videos
hard_images = get_images_from_videos(hard_videos, num_samples=2)

target_images = easy_images + hard_images

if len(target_images) == 0:
    print(f" ERROR: No images found. Check your video names and make sure they match the filenames exactly (without the frame number)!")
else:
    print(f"Found {len(target_images)} target images. Generating side-by-side visuals...\n")
    
generate_comparison_figure(
    image_paths=target_images,
    yolo_path=YOLO_WEIGHTS,
    frcnn_path=FRCNN_WEIGHTS,
    output_dir='/kaggle/working/report_visuals'
)

# DETR

In [ ]:
!pip install transformers timm

In [ ]:
import torch
from transformers import DeformableDetrForObjectDetection
from PIL import Image
from transformers import AutoImageProcessor
from torchmetrics.detection.mean_ap import MeanAveragePrecision

import os
from tqdm.auto import tqdm

In [ ]:
def build_deformable_detr(experiment="Pretrained", num_classes=3):
    """
    Loads Deformable DETR and applies the freezing strategies for Task 3.
    """
    print(f"\nLoading Deformable DETR for {experiment}...")
    
    # Load the pretrained model from Hugging Face
    model = DeformableDetrForObjectDetection.from_pretrained(
        "SenseTime/deformable-detr",
        num_labels=num_classes,
        ignore_mismatched_sizes=True
    )
    
    if experiment == "Pretrained":
        model.eval()
        return model

    # --- FINE-TUNING STRATEGIES (7.1.2) ---
    
    # 1. ALWAYS freeze the ResNet Backbone. 
    for param in model.model.backbone.parameters():
        param.requires_grad = False

    # 2. Apply Experiment-Specific Freezing
    if experiment == "Exp1":
        print("Strategy: Fine-tuning FULL Transformer (Encoder + Decoder)")
        # Do nothing. Encoder and Decoder are trainable by default.
        
    elif experiment == "Exp2":
        print("Strategy: Fine-tuning DECODER ONLY")
        # Freeze the Encoder
        for param in model.model.encoder.parameters():
            param.requires_grad = False
            
    elif experiment == "Exp3":
        print("Strategy: Fine-tuning ENCODER ONLY")
        # Freeze the Decoder
        for param in model.model.decoder.parameters():
            param.requires_grad = False
            
    # 3. VERIFICATION CHECK (THE FIX)
    # Ensure the classification and bounding box heads are ALWAYS trainable.
    # In Deformable DETR, these are named class_embed and bbox_embed!
    for param in model.class_embed.parameters():
        param.requires_grad = True
    for param in model.bbox_embed.parameters():
        param.requires_grad = True
        
    # Print a summary of trainable parameters
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable Parameters: {trainable_params:,} / {total_params:,} ({(trainable_params/total_params)*100:.2f}%)")
    
    return model

In [ ]:
def format_detr_labels(targets, batched_images):
    """
    Translates Faster R-CNN absolute [x1, y1, x2, y2] bounding boxes 
    into DETR normalized [center_x, center_y, width, height] boxes.
    """
    detr_labels = []
    # batched_images shape is [Batch, Channels, Height, Width]
    h, w = batched_images.shape[-2], batched_images.shape[-1]
    
    for t in targets:
        boxes = t["boxes"] # Absolute [x_min, y_min, x_max, y_max]
        
        if len(boxes) > 0:
            # Convert corners to center, width, height
            box_width = boxes[:, 2] - boxes[:, 0]
            box_height = boxes[:, 3] - boxes[:, 1]
            center_x = boxes[:, 0] + (box_width / 2)
            center_y = boxes[:, 1] + (box_height / 2)
            
            # Normalize by image dimensions
            norm_cx = center_x / w
            norm_cy = center_y / h
            norm_w = box_width / w
            norm_h = box_height / h
            
            # Stack into expected shape and clamp to [0, 1] to prevent loss spikes
            norm_boxes = torch.stack([norm_cx, norm_cy, norm_w, norm_h], dim=-1)
            norm_boxes = torch.clamp(norm_boxes, 0.0, 1.0)
        else:
            norm_boxes = torch.empty((0, 4), dtype=torch.float32)
            
        detr_labels.append({
            "class_labels": t["labels"],
            "boxes": norm_boxes
        })
        
    return detr_labels

In [ ]:
def train_detr_epoch(model, optimizer, data_loader, device, epoch):
    model.train()
    total_loss_this_epoch = 0.0
    
    loop = tqdm(data_loader, desc=f"Epoch [{epoch}]", leave=True)
    
    for batch_idx, (images, targets) in enumerate(loop):
        
        # Identify the max dimensions in this specific batch
        max_h = max([img.shape[1] for img in images])
        max_w = max([img.shape[2] for img in images])
        
        # Pad each image so they are all the same size and stack them
        batched_imgs = []
        for img in images:
            pad_w = max_w - img.shape[2]
            pad_h = max_h - img.shape[1]
            # F.pad format is (padding_left, padding_right, padding_top, padding_bottom)
            padded_img = F.pad(img, (0, pad_w, 0, pad_h)) 
            batched_imgs.append(padded_img)
            
        # This creates a single 4D tensor: [Batch, Channels, Height, Width]
        images_tensor = torch.stack(batched_imgs).to(device)
        
        # Translate and move labels to GPU (now using images_tensor for height/width)
        detr_labels = format_detr_labels(targets, images_tensor)
        detr_labels = [{k: v.to(device) for k, v in t.items()} for t in detr_labels]
        
        # Forward Pass
        # We pass images_tensor here
        outputs = model(pixel_values=images_tensor, labels=detr_labels)
        
        # If you are using DataParallel, use outputs.loss.mean()
        # If single GPU, use outputs.loss
        loss = outputs.loss 
        
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping for Transformer stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)
        
        optimizer.step()
        
        total_loss_this_epoch += loss.item()
        loop.set_postfix(loss=f"{loss.item():.4f}")
            
    return total_loss_this_epoch / len(data_loader)

In [ ]:
def run_all_detr_experiments(num_epochs=10):
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    print(f"Training on device: {device}")
    
    # 1. Setup Data
    train_img_dir = '/kaggle/working/yolo_birdsai/images/train'
    train_label_dir = '/kaggle/working/yolo_birdsai/labels/train'
    dataset = BIRDSAIYOLODataset(train_img_dir, train_label_dir)
    data_loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=2, collate_fn=collate_fn)
    
    # 2. Define the Experiments from the Rubric
    experiments = ["Exp1", "Exp2", "Exp3"]
    all_loss_histories = {}
    
    for exp in experiments:
        print("\n" + "="*50)
        print(f"🚀 INITIATING {exp.upper()} TRAINING")
        print("="*50)
        
        # Build model with specific freezing strategy
        model = build_deformable_detr(experiment=exp, num_classes=3)
        model.to(device)
        
        # Setup Optimizer: The rubric explicitly requires AdamW.
        # CRITICAL: We MUST use filter() so the optimizer only tries to update 
        # the parameters we left un-frozen in our build function.
        trainable_params = filter(lambda p: p.requires_grad, model.parameters())
        optimizer = torch.optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-4)
        
        loss_history = []
        
        for epoch in range(1, num_epochs + 1):
            epoch_loss = train_detr_epoch(model, optimizer, data_loader, device, epoch)
            loss_history.append(epoch_loss)
            
        # Save the model weights
        save_path = f'/kaggle/working/deformable_detr_{exp.lower()}.pth'
        torch.save(model.state_dict(), save_path)
        print(f"\n✅ {exp} Complete! Weights saved to {save_path}")
        
        all_loss_histories[exp] = loss_history
        
    return all_loss_histories

In [ ]:
detr_histories = run_all_detr_experiments(num_epochs=5)

In [ ]:
def plot_detr_training_curves(histories, output_dir='/kaggle/working'):
    """
    Takes the dictionary of loss histories from the 3 experiments
    and plots them on a single graph for easy comparison.
    """
    print("\n--- Generating Training Curves for Report ---")
    
    plt.figure(figsize=(10, 6))
    
    # Custom styling for the three experiments
    styles = {
        "Exp1": {"color": "blue", "label": "Exp1: Full Model", "marker": "o"},
        "Exp2": {"color": "green", "label": "Exp2: Decoder Only", "marker": "s"},
        "Exp3": {"color": "red", "label": "Exp3: Encoder Only", "marker": "^"}
    }
    
    for exp, losses in histories.items():
        if not losses:
            continue # Skip if empty
            
        epochs = range(1, len(losses) + 1)
        
        plt.plot(
            epochs, 
            losses, 
            color=styles[exp]["color"], 
            label=styles[exp]["label"], 
            marker=styles[exp]["marker"],
            linewidth=2,
            markersize=8
        )

    plt.title('Deformable DETR Fine-Tuning Convergence', fontsize=16, fontweight='bold')
    plt.xlabel('Epoch', fontsize=14)
    plt.ylabel('SetCriterion Loss', fontsize=14)
    
    # Force x-axis to show whole integer epochs only
    plt.xticks(range(1, len(next(iter(histories.values()))) + 1))
    
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(fontsize=12)
    
    # Save the plot for the report
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, 'task3_training_curves.png')
    plt.savefig(save_path, bbox_inches='tight', dpi=150)
    
    # Display it in the Kaggle notebook
    plt.show()
    plt.close()
    
    print(f"✅ Training curves saved to: {save_path}")

In [ ]:
if 'detr_histories' in locals() or 'detr_histories' in globals():
    plot_detr_training_curves(detr_histories)
else:
    print("Warning: 'detr_histories' not found. Run the training cell first!")

## Quantitative Analysis of DETR

In [ ]:

# The Evaluation Engine
def evaluate_single_detr(model_name, experiment_type="Exp1", device='cuda'):
    if model_name == "Pretrained":
        model = build_deformable_detr("Pretrained")
    else:
        model = build_deformable_detr(experiment_type)
        weights_path = f'{detr_weights_path}/deformable_detr_{experiment_type.lower()}.pth'
        if os.path.exists(weights_path):
            model.load_state_dict(torch.load(weights_path, map_location=device))
        else:
            print(f"⚠️ Warning: Weights for {experiment_type} not found at {weights_path}. Skipping.")
            return None

    model.to(device)
    model.eval()

    # The standalone Image Processor fix
    processor = AutoImageProcessor.from_pretrained("SenseTime/deformable-detr")

    test_img_dir = '/kaggle/working/yolo_birdsai/images/test'
    test_label_dir = '/kaggle/working/yolo_birdsai/labels/test'
    test_dataset = BIRDSAIYOLODataset(test_img_dir, test_label_dir)
    test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)

    metric_animals = MeanAveragePrecision(iou_type="bbox", iou_thresholds=[0.5])
    metric_humans = MeanAveragePrecision(iou_type="bbox", iou_thresholds=[0.5])

    with torch.no_grad():
        for images, targets in tqdm(test_loader, desc=f"Testing {model_name}"):
            if isinstance(images, (list, tuple)):
                max_h = max([img.shape[1] for img in images])
                max_w = max([img.shape[2] for img in images])
                images = torch.stack([F.pad(img, (0, max_w - img.shape[2], 0, max_h - img.shape[1])) for img in images])
            
            images = images.to(device)
            h, w = images.shape[-2], images.shape[-1]
            
            outputs = model(pixel_values=images)
            
            target_sizes = torch.tensor([[h, w]] * images.shape[0]).to(device)
            results = processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=0.0)

            for i in range(len(results)):
                res = results[i]
                target = targets[i]
                
                p_boxes = res['boxes'].cpu()
                p_scores = res['scores'].cpu()
                p_labels = res['labels'].cpu()
                
                t_boxes = target['boxes'].cpu()
                t_labels = target['labels'].cpu()

                # Animals (1)
                a_pred_mask = p_labels == 1
                a_targ_mask = t_labels == 1
                metric_animals.update(
                    [{"boxes": p_boxes[a_pred_mask], "scores": p_scores[a_pred_mask], "labels": p_labels[a_pred_mask]}],
                    [{"boxes": t_boxes[a_targ_mask], "labels": t_labels[a_targ_mask]}]
                )

                # Humans (2)
                h_pred_mask = p_labels == 2
                h_targ_mask = t_labels == 2
                metric_humans.update(
                    [{"boxes": p_boxes[h_pred_mask], "scores": p_scores[h_pred_mask], "labels": p_labels[h_pred_mask]}],
                    [{"boxes": t_boxes[h_targ_mask], "labels": t_labels[h_targ_mask]}]
                )

    return {"animals": metric_animals.compute(), "humans": metric_humans.compute()}

# 2. The Table Generator
def generate_table_3():
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    model_configs = [
        ("Pretrained", "Pretrained"),
        ("Exp1", "Exp1"),
        ("Exp2", "Exp2"),
        ("Exp3", "Exp3")
    ]
    
    results_master = {}
    
    for label, config in model_configs:
        res = evaluate_single_detr(label, config, device)
        if res:
            results_master[label] = res

    # Print Table
    header = f"{'Scale / Category':<20} | {'Pretrained':<10} | {'Exp1':<10} | {'Exp2':<10} | {'Exp3':<10}"
    print("\n" + "="*75)
    print("📊 TABLE 3: DEFORMABLE DETR QUANTITATIVE COMPARISON")
    print("="*75)
    print(header)
    print("-" * 75)

    rows = [
        ("SA (Small Animals)", "animals", "map_small"),
        ("MA (Medium Animals)", "animals", "map_medium"),
        ("LA (Large Animals)", "animals", "map_large"),
        ("Animals (Overall)", "animals", "map_50"),
        ("SH (Small Humans)", "humans", "map_small"),
        ("MH (Medium Humans)", "humans", "map_medium"),
        ("LH (Large Humans)", "humans", "map_large"),
        ("Humans (Overall)", "humans", "map_50"),
    ]

    for row_name, cat, metric_key in rows:
        row_str = f"{row_name:<20}"
        for label, _ in model_configs:
            if label in results_master:
                val = results_master[label][cat][metric_key].item()
                row_str += f" | {val:>10.4f}"
            else:
                row_str += f" | {'N/A':>10}"
        print(row_str)
    
    overall_row = f"{'Overall':<20}"
    for label, _ in model_configs:
        if label in results_master:
            a_score = results_master[label]['animals']['map_50'].item()
            h_score = results_master[label]['humans']['map_50'].item()
            valid_scores = [s for s in [a_score, h_score] if s >= 0]
            avg = sum(valid_scores) / len(valid_scores) if valid_scores else 0.0
            overall_row += f" | {avg:>10.4f}"
        else:
            overall_row += f" | {'N/A':>10}"
    
    print("-" * 75)
    print(overall_row)
    print("="*75)

# 3. Execution Trigger
if __name__ == "__main__":
    generate_table_3()
    

## Qualitative Analysis of DETR

In [ ]:
import random
from transformers import AutoImageProcessor

easy_videos = ['0000000352_0000000000'] 
hard_videos = ['0000000058_0000000000'] 
test_dir = '/kaggle/working/yolo_birdsai/images/test'

DETR_WEIGHTS = {
    "Pretrained": "Pretrained", # Loads from HuggingFace
    "Exp1": "/kaggle/input/models/zaiemgulzar/detr/pytorch/default/1/deformable_detr_exp1.pth",
    "Exp2": "/kaggle/input/models/zaiemgulzar/detr/pytorch/default/1/deformable_detr_exp2.pth",
    "Exp3": "/kaggle/input/models/zaiemgulzar/detr/pytorch/default/1/deformable_detr_exp3.pth",
}

def get_target_frames(video_list, num_per_video=2):
    frames = []
    for vid in video_list:
        matches = glob.glob(os.path.join(test_dir, f"{vid}_*.jpg"))
        if matches:
            frames.extend(random.sample(matches, min(num_per_video, len(matches))))
    return frames

def generate_detr_report_visuals(image_paths, conf_threshold=0.30):
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    processor = AutoImageProcessor.from_pretrained("SenseTime/deformable-detr")
    
    # Load all models once to save time
    models = {}
    for exp_name, path in DETR_WEIGHTS.items():
        print(f"Loading {exp_name}...")
        m = build_deformable_detr(exp_name)
        if exp_name != "Pretrained" and os.path.exists(path):
            m.load_state_dict(torch.load(path, map_location=device))
        m.to(device).eval()
        models[exp_name] = m

    class_names = {1: "Animal", 2: "Human"}
    num_imgs = len(image_paths)
    fig, axes = plt.subplots(num_imgs, 4, figsize=(24, 6 * num_imgs))
    
    # Ensure axes is 2D even if only 1 image
    if num_imgs == 1: axes = [axes]

    for row, img_path in enumerate(image_paths):
        img_name = os.path.basename(img_path)
        pil_img = Image.open(img_path).convert("RGB")
        img_tensor = TF.to_tensor(pil_img)
        h, w = img_tensor.shape[1], img_tensor.shape[2]
        base_uint8 = (img_tensor * 255).to(torch.uint8)

        for col, (exp_name, model) in enumerate(models.items()):
            # Run Inference
            with torch.no_grad():
                outputs = model(pixel_values=img_tensor.unsqueeze(0).to(device))
                results = processor.post_process_object_detection(outputs, target_sizes=torch.tensor([[h, w]]).to(device), threshold=conf_threshold)[0]
            
            p_boxes = results['boxes'].cpu()
            p_labels = results['labels'].cpu()
            p_scores = results['scores'].cpu()
            
            # Label lookup (safeguard for Pretrained COCO labels)
            labels_str = [f"{class_names.get(l.item(), 'Misc')}: {s:.2f}" for l, s in zip(p_labels, p_scores)]
            
            # Draw (Red for predictions)
            draw_img = base_uint8.clone()
            if len(p_boxes) > 0:
                draw_img = draw_bounding_boxes(draw_img, p_boxes, labels=labels_str, colors="red", width=3, font_size=18)
            
            axes[row][col].imshow(draw_img.permute(1, 2, 0).numpy())
            axes[row][col].axis('off')
            
            if row == 0:
                axes[row][col].set_title(f"{exp_name}", fontsize=20, fontweight='bold')
        
        # Add side-label for difficulty
        difficulty = "EASY" if any(v in img_name for v in easy_videos) else "HARD"
        axes[row][0].text(-40, h//2, f"{difficulty}\n{img_name}", rotation=90, va='center', fontweight='bold', fontsize=12)

    plt.tight_layout()
    plt.show()

target_images = get_target_frames(easy_videos, 2) + get_target_frames(hard_videos, 2)
generate_detr_report_visuals(target_images)

## Comparisons with YOLO and FRCNN

In [ ]:
import random

WEIGHTS = {
    "YOLO": "/kaggle/input/models/zaiemgulzar/yolo/pytorch/default/1/best.pt", # Or runs/train/.../best.pt
    "FRCNN_CE": "/kaggle/input/models/zaiemgulzar/frcnn/pytorch/default/1/frcnn_ce_weights.pth",
    "FRCNN_FL": "/kaggle/input/models/zaiemgulzar/frcnn/pytorch/default/1/frcnn_fl_weights.pth",
    "DETR_PRE": "Pretrained",
    "DETR_EXP1": "/kaggle/input/models/zaiemgulzar/detr/pytorch/default/1/deformable_detr_exp1.pth",
    "DETR_EXP2": "/kaggle/input/models/zaiemgulzar/detr/pytorch/default/1/deformable_detr_exp2.pth",
    "DETR_EXP3": "/kaggle/input/models/zaiemgulzar/detr/pytorch/default/1/deformable_detr_exp3.pth",
}

test_dir = '/kaggle/working/yolo_birdsai/images/test'
test_label_dir = '/kaggle/working/yolo_birdsai/labels/test'

easy_videos = ['0000000352_0000000000'] 
hard_videos = ['0000000058_0000000000'] 


def get_images_from_videos(video_list, num_samples=5):
    """Finds all frames belonging to specific videos and samples a few."""
    collected_frames = []
    for video_name in video_list:
        search_pattern = os.path.join(test_dir, f"{video_name}_*.jpg")
        frames = glob.glob(search_pattern)
        collected_frames.extend(frames)
        
    if len(collected_frames) == 0:
        print(f"Warning: Found 0 images for videos: {video_list}")
        return []
        
    return random.sample(collected_frames, min(num_samples, len(collected_frames)))

def generate_ultimate_master_comparison(image_paths, output_dir='/kaggle/working/report_ultimate', conf_threshold=0.30):
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    os.makedirs(output_dir, exist_ok=True)
    
    print("Loading all models into memory (This may take a moment)...")
    models = {}
    
    # Load YOLO
    if os.path.exists(WEIGHTS["YOLO"]):
        models["YOLO"] = YOLO(WEIGHTS["YOLO"])
    else: print("YOLO weights missing.")

    # Load FRCNNs (Assumes your build functions are defined!)
    if os.path.exists(WEIGHTS["FRCNN_CE"]):
        models["FRCNN_CE"] = build_custom_frcnn_base(num_classes=3)
        models["FRCNN_CE"].load_state_dict(torch.load(WEIGHTS["FRCNN_CE"], map_location=device))
        models["FRCNN_CE"].to(device).eval()
    
    if os.path.exists(WEIGHTS["FRCNN_FL"]):
        models["FRCNN_FL"] = build_frcnn_focal_loss(num_classes=3)
        models["FRCNN_FL"].load_state_dict(torch.load(WEIGHTS["FRCNN_FL"], map_location=device))
        models["FRCNN_FL"].to(device).eval()

    # Load DETRs
    detr_processor = AutoImageProcessor.from_pretrained("SenseTime/deformable-detr")
    detr_exps = [("DETR_PRE", "Pretrained"), ("DETR_EXP1", "Exp1"), ("DETR_EXP2", "Exp2"), ("DETR_EXP3", "Exp3")]
    
    for key, exp_type in detr_exps:
        m = build_deformable_detr(exp_type)
        if exp_type != "Pretrained" and os.path.exists(WEIGHTS[key]):
            m.load_state_dict(torch.load(WEIGHTS[key], map_location=device))
        m.to(device).eval()
        models[key] = m

    class_names = {1: "Animal", 2: "Human"}
    yolo_class_names = {0: "Animal", 1: "Human"}
    
    # Helper for pretrained COCO classes
    def get_class_name(label_id):
        return class_names.get(label_id, f"COCO_{label_id}")

    test_dataset = BIRDSAIYOLODataset(test_dir, test_label_dir)

    for img_path in image_paths:
        if not os.path.exists(img_path):
            continue
            
        img_name = os.path.basename(img_path)
        print(f"\nProcessing {img_name}...")
        
        # Match image back to dataset to get exact Ground Truth
        try:
            idx = test_dataset.img_names.index(img_name)
            img_tensor, target = test_dataset[idx]
        except ValueError:
            print(f"Skipping {img_name}, not found in dataset mapping.")
            continue

        base_img_uint8 = (img_tensor * 255).to(torch.uint8)
        h, w = img_tensor.shape[1], img_tensor.shape[2]
        drawn_images = {}

        # GROUND TRUTH 
        gt_boxes, gt_labels = target['boxes'], target['labels']
        gt_strings = [f"GT {class_names[l.item()]}" for l in gt_labels]
        drawn_images["GT"] = base_img_uint8.clone()
        if len(gt_boxes) > 0:
            drawn_images["GT"] = draw_bounding_boxes(drawn_images["GT"], gt_boxes, labels=gt_strings, colors="green", width=3, font_size=18)

        # YOLO 
        drawn_images["YOLO"] = base_img_uint8.clone()
        if "YOLO" in models:
            y_res = models["YOLO"](img_path, verbose=False)[0]
            y_keep = y_res.boxes.conf.cpu() > conf_threshold
            y_boxes = y_res.boxes.xyxy.cpu()[y_keep]
            y_strings = [f"{yolo_class_names[l.item()]}: {s:.2f}" for l, s in zip(y_res.boxes.cls.cpu().int()[y_keep], y_res.boxes.conf.cpu()[y_keep])]
            if len(y_boxes) > 0: drawn_images["YOLO"] = draw_bounding_boxes(drawn_images["YOLO"], y_boxes, labels=y_strings, colors="red", width=3, font_size=18)

        # FRCNN Inferences
        for m_key, color in [("FRCNN_CE", "blue"), ("FRCNN_FL", "orange")]:
            drawn_images[m_key] = base_img_uint8.clone()
            if m_key in models:
                with torch.no_grad(): f_res = models[m_key]([img_tensor.to(device)])[0]
                f_keep = f_res['scores'] > conf_threshold
                f_boxes, f_labels, f_scores = f_res['boxes'][f_keep].cpu(), f_res['labels'][f_keep].cpu(), f_res['scores'][f_keep].cpu()
                f_strings = [f"{class_names[l.item()]}: {s:.2f}" for l, s in zip(f_labels, f_scores)]
                if len(f_boxes) > 0: drawn_images[m_key] = draw_bounding_boxes(drawn_images[m_key], f_boxes, labels=f_strings, colors=color, width=3, font_size=18)

        # DETR Inferences
        for m_key, color in [("DETR_PRE", "gray"), ("DETR_EXP1", "magenta"), ("DETR_EXP2", "purple"), ("DETR_EXP3", "cyan")]:
            drawn_images[m_key] = base_img_uint8.clone()
            if m_key in models:
                with torch.no_grad():
                    d_out = models[m_key](pixel_values=img_tensor.unsqueeze(0).to(device))
                    d_res = detr_processor.post_process_object_detection(d_out, target_sizes=torch.tensor([[h, w]]).to(device), threshold=conf_threshold)[0]
                d_boxes, d_labels, d_scores = d_res['boxes'].cpu(), d_res['labels'].cpu(), d_res['scores'].cpu()
                d_strings = [f"{get_class_name(l.item())}: {s:.2f}" for l, s in zip(d_labels, d_scores)]
                if len(d_boxes) > 0: drawn_images[m_key] = draw_bounding_boxes(drawn_images[m_key], d_boxes, labels=d_strings, colors=color, width=3, font_size=18)

        # PLOTTING THE 2x4 GRID
        fig, axes = plt.subplots(2, 4, figsize=(28, 12))
        axes = axes.flatten()
        
        plot_order = [
            ("GT", "Ground Truth", "green"), ("YOLO", "YOLOv8", "red"), ("FRCNN_CE", "Faster R-CNN (Cross Entropy)", "blue"), ("FRCNN_FL", "Faster R-CNN (Focal Loss)", "orange"),
            ("DETR_PRE", "Deformable DETR (Pretrained)", "gray"), ("DETR_EXP1", "DETR Exp1 (Full Fine-tune)", "magenta"), ("DETR_EXP2", "DETR Exp2 (Decoder Only)", "purple"), ("DETR_EXP3", "DETR Exp3 (Encoder Only)", "cyan")
        ]

        for idx, (key, title, color) in enumerate(plot_order):
            axes[idx].imshow(drawn_images[key].permute(1, 2, 0).numpy())
            axes[idx].set_title(title, fontsize=18, fontweight='bold', color=color)
            axes[idx].axis('off')

        plt.tight_layout()
        difficulty = "EASY" if "0000000352" in img_name else "HARD"
        fig.suptitle(f"Master Overview ({difficulty}) | {img_name} | Conf > {conf_threshold}", fontsize=24, fontweight='bold', y=1.03)
        
        save_path = os.path.join(output_dir, f"master_{difficulty}_{img_name}")
        plt.savefig(save_path, bbox_inches='tight', dpi=150)
        plt.show()
        plt.close(fig)

    print(f"\n✅ All master grids displayed and saved to: {output_dir}")

In [ ]:
print("Gathering Easy Images...")
easy_images = get_images_from_videos(easy_videos, num_samples=2)

print("Gathering Hard Images...")
hard_images = get_images_from_videos(hard_videos, num_samples=2)

target_images = easy_images + hard_images

if len(target_images) == 0:
    print("ERROR: No images found.")
else:
    print(f"Found {len(target_images)} target images. Initiating Master Grid Generation...\n")
    # You can adjust conf_threshold if the models are being too noisy or too strict
    generate_ultimate_master_comparison(target_images, conf_threshold=0.15)